In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import scanpy as sc
from collections import Counter
from scipy import sparse
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

import warnings
warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

In [14]:
# Main loop for enrichment analysis
samples = ["Xenium_5K_BC", "Xenium_5K_OC", "Xenium_5K_CC", "Xenium_5K_LC", "Xenium_5K_Prostate"]
pseudocount = 0.1
n_top_hvg = 2500       # HVGs per sample (seurat_v3 on whole-cell tumor counts); 5001 genes on the panel
min_hvg_samples = 2    # a gene must be HVG in at least this many samples
target_genes_dict = {}
hvg_dict = {}

for sample in samples:
    
    print("="* 20 + f"Processing sample: {sample}" + "="* 20)
    
    # Paths
    intermediate_path = f"../../data/{sample}/intermediate_data/"
    processed_path = f"../../data/{sample}/processed_data/"
    output_path = f"../../output/1_exploration/{sample}/"
    os.makedirs(output_path, exist_ok=True)

    # Read data
    adata = sc.read_h5ad(intermediate_path + "adata.h5ad")
    cell_ids = np.load(processed_path + "cell_ids.npy", allow_pickle=True)
    assert adata.shape[0] > cell_ids.shape[0], "cell_ids should be a subset of adata.obs_names"

    # Tumor cells
    adata_tumor = adata[adata.obs["cell_id"].isin(cell_ids)].copy()

    # Highly variable genes of the tumor cells (whole-cell raw counts, independent of compartment)
    sc.pp.highly_variable_genes(adata_tumor, flavor="seurat_v3", n_top_genes=n_top_hvg)
    hvg_dict[sample] = set(adata_tumor.var_names[adata_tumor.var["highly_variable"]])
    print(f"Number of highly variable genes: {len(hvg_dict[sample])}")

    # Read nuclear and cytoplasmic expression matrices
    X_nuc = sparse.load_npz(processed_path + "nuclear_expression_matrix.npz")
    X_cyto = sparse.load_npz(processed_path + "cytoplasmic_expression_matrix.npz")

    assert adata_tumor.shape[0] == X_nuc.shape[0], "adata_tumor and X_nuc should have the same number of cells"
    assert X_nuc.shape[0] == X_cyto.shape[0], "Nuclear and cytoplasmic expression matrices should have the same number of cells"
    assert X_nuc.shape[1] == X_cyto.shape[1], "Nuclear and cytoplasmic expression matrices should have the same number of genes"
    
    # Compute cytoplasmic fraction
    nuc_counts = np.asarray(X_nuc.sum(axis=0)).ravel()
    cyto_counts = np.asarray(X_cyto.sum(axis=0)).ravel()
    cyto_fraction = cyto_counts / (cyto_counts + nuc_counts + 1e-12)

    # Compute log2 fold change between cytoplasmic and nuclear counts
    nuc_total = nuc_counts.sum()
    cyto_total = cyto_counts.sum()

    nuc_cpm = nuc_counts / nuc_total * 1e6
    cyto_cpm = cyto_counts / cyto_total * 1e6
    log2fc = np.log2((cyto_cpm + pseudocount) / (nuc_cpm + pseudocount))
    
    # Compute library sizes, normalize the expression matrices, log transform, and convert to CSC format for efficient column slicing
    nuc_libsize = np.asarray(X_nuc.sum(axis=1)).ravel()
    cyto_libsize = np.asarray(X_cyto.sum(axis=1)).ravel()

    nuc_norm = X_nuc.multiply(1e4 / np.maximum(nuc_libsize, 1)[:, None]).tocsr()
    cyto_norm = X_cyto.multiply(1e4 / np.maximum(cyto_libsize, 1)[:, None]).tocsr()

    nuc_log = nuc_norm.copy()
    cyto_log = cyto_norm.copy()

    nuc_log.data = np.log1p(nuc_log.data)
    cyto_log.data = np.log1p(cyto_log.data)
    
    nuc_log_csc = nuc_log.tocsc()
    cyto_log_csc = cyto_log.tocsc()

    # Perform Wilcoxon signed-rank test for each gene
    n_genes = nuc_log_csc.shape[1]
    stats = np.full(n_genes, np.nan)
    pvalues = np.full(n_genes, np.nan)

    for j in range(n_genes):
        x_nuc = nuc_log_csc[:, j].toarray().ravel()
        x_cyto = cyto_log_csc[:, j].toarray().ravel()
        diff = x_cyto - x_nuc
        if np.all(diff == 0):
            stats[j] = 0
            pvalues[j] = 1
        else:
            res = wilcoxon(x_cyto, x_nuc, alternative="two-sided")
            stats[j] = res.statistic
            pvalues[j] = res.pvalue
    fdr = multipletests(pvalues, method="fdr_bh")[1]
    
    # Create a DataFrame to store the results
    result = pd.DataFrame({
        "gene": adata_tumor.var_names,
        "nuclear_counts": nuc_counts,
        "cytoplasmic_counts": cyto_counts,
        "total_counts": nuc_counts + cyto_counts,
        "cytoplasmic_fraction": cyto_fraction,
        "nuclear_CPM": nuc_cpm,
        "cytoplasmic_CPM": cyto_cpm,
        "log2FC_cyto_vs_nuc": log2fc,
        "wilconxon_stat": stats,
        "pvalue": pvalues,
        "FDR": fdr,
        "highly_variable": adata_tumor.var["highly_variable"].values
    })

    # Filter out genes with low total counts (bottom 10%)
    total_count_thr = np.percentile(result["total_counts"], 10)  # Filter out genes with low total counts (bottom 10%)
    print(f"Filtering out genes with total counts below {total_count_thr:.2f} (bottom 10%)")

    result = result[result["total_counts"] >= total_count_thr]
    result = result.sort_values("log2FC_cyto_vs_nuc", ascending=False)
    result.to_csv(output_path + "enrichment_results.csv", index=False)
    
    # Identify genes enriched in cytoplasm based on log2 fold change and FDR (HVG filter applied after the loop)
    criteria = (result["log2FC_cyto_vs_nuc"] > 0) & (result["FDR"] < 0.05)
    target_genes_dict[sample] = result[criteria]["gene"].tolist()
    print(f"Number of genes enriched in cytoplasm: {result[criteria].shape[0]}")

====================Processing sample: Xenium_5K_BC====================
Number of highly variable genes: 2500
Filtering out genes with total counts below 335.00 (bottom 10%)
Number of genes enriched in cytoplasm: 1158
====================Processing sample: Xenium_5K_OC====================
Number of highly variable genes: 2500
Filtering out genes with total counts below 536.00 (bottom 10%)
Number of genes enriched in cytoplasm: 1053
====================Processing sample: Xenium_5K_CC====================
Number of highly variable genes: 2500
Filtering out genes with total counts below 259.00 (bottom 10%)
Number of genes enriched in cytoplasm: 1244
====================Processing sample: Xenium_5K_LC====================
Number of highly variable genes: 2500
Filtering out genes with total counts below 206.00 (bottom 10%)
Number of genes enriched in cytoplasm: 1612
====================Processing sample: Xenium_5K_Prostate====================
Number of highly variable genes: 2500
Filtering ou

## Gene selection criteria

A gene is retained if it passes both filters:

1. **Highly variable** (`seurat_v3`, top `n_top_hvg = 2000` of the 5001-gene panel, on the whole-cell raw counts of the tumor cells) in at least `min_hvg_samples = 2` of the five samples. This removes genes that are cytoplasm-enriched everywhere but do not vary across tumor cells and therefore carry no subtype information.
2. **Cytoplasm-enriched** (log2FC > 0, FDR < 0.05, after dropping the bottom 10% by total counts) in at least 3 of the five samples.

The HVG filter is applied first, so the overlap plot below and the saved `overlap_genes_3plus_samples.npy` describe the final gene set.

In [15]:
# HVG filter: keep genes that are highly variable in at least `min_hvg_samples` samples,
# applied before the overlap counting so the overlap plot and the saved file describe the final gene set
hvg_counts = Counter()
for sample in samples:
    hvg_counts.update(hvg_dict[sample])
hvg_genes = {gene for gene, count in hvg_counts.items() if count >= min_hvg_samples}
print(f"Number of genes HVG in at least {min_hvg_samples} samples: {len(hvg_genes)}")

for sample in samples:
    before = len(target_genes_dict[sample])
    target_genes_dict[sample] = [gene for gene in target_genes_dict[sample] if gene in hvg_genes]
    print(f"{sample}: cytoplasm-enriched genes {before} -> {len(target_genes_dict[sample])} after HVG filter")

Number of genes HVG in at least 2 samples: 3318
Xenium_5K_BC: cytoplasm-enriched genes 1158 -> 658 after HVG filter
Xenium_5K_OC: cytoplasm-enriched genes 1053 -> 616 after HVG filter
Xenium_5K_CC: cytoplasm-enriched genes 1244 -> 737 after HVG filter
Xenium_5K_LC: cytoplasm-enriched genes 1612 -> 1008 after HVG filter
Xenium_5K_Prostate: cytoplasm-enriched genes 1207 -> 670 after HVG filter


In [16]:
# Overlap genes across at least three samples
gene_counts = Counter()
for sample in samples:
    gene_counts.update(set(target_genes_dict[sample]))
overlap_genes = {gene for gene, count in gene_counts.items() if count >= 3}

enrichment_path = "../../output/1_exploration/enrichment/"
os.makedirs(enrichment_path, exist_ok=True)
np.save(enrichment_path + "overlap_genes_3plus_samples.npy", np.array(list(overlap_genes)))
print(f"Number of genes overlapping in at least 3 samples: {len(overlap_genes)}")

Number of genes overlapping in at least 3 samples: 634


In [ ]:
# Overlap gene counts (after the HVG filter, i.e. the final gene universe)
gene_counts = Counter()
for sample in samples:
    gene_counts.update(set(target_genes_dict[sample]))
counts = [sum(count >= i + 1 for count in gene_counts.values()) for i in range(len(samples))]

plt.figure(figsize=(7, 6))
plt.ylim(0, 2500)
plt.plot(range(1, len(samples) + 1), counts)
plt.scatter(range(1, len(samples) + 1), counts, s=50)
plt.xlabel("Number of Samples")
plt.ylabel("Number of Overlapping Genes")
plt.title(f"Cytoplasm-enriched HVGs (HVG in >= {min_hvg_samples} samples) shared across samples")
plt.xticks(range(1, len(samples) + 1))
plt.savefig(enrichment_path + "overlap_genes_counts.png", dpi=300, bbox_inches="tight")
plt.close()